# RAG 评估报告 | Hexa-Ops

评估混合检索器（FAISS + BM25 + RRF融合）在告警运维知识库上的性能。

## 评估指标
- **Precision@K**：Top-K 结果中相关文档的比例
- **Recall@K**：Top-K 结果中检索到的相关文档占全部相关的比例
- **MRR**（Mean Reciprocal Rank）：第一个相关结果排名的倒数均值

## 对比方案
| 方案 | 说明 |
|------|------|
| Hybrid (RRF) | FAISS + BM25 倒数秩融合 |
| FAISS Only | 纯向量检索 |
| BM25 Only | 纯关键词检索 |

In [ ]:
import sys, os
# 将 Agent_Rag_Ops 加入路径
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('项目路径:', PROJECT_ROOT)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans', 'SimHei', 'WenQuanYi Micro Hei']
matplotlib.rcParams['axes.unicode_minus'] = False

from typing import List, Dict, Any
print('依赖导入完成')

## 1. Ground Truth 数据集

每条数据：查询 query + 期望包含该内容的文档关键词列表（relevance_keywords）。  
若 retrieved 文档的 `page_content` 包含任意一个关键词，则认为该文档相关。

In [ ]:
# Ground Truth QA 数据集
# relevance_keywords: 文档中应包含的关键词（任意命中即为相关）
GROUND_TRUTH = [
    {
        "query": "Pod OOMKilled 如何处理",
        "relevance_keywords": ["OOMKilled", "内存限制", "resources.limits", "内存"],
        "description": "Pod 内存溢出处理"
    },
    {
        "query": "服务 CPU 使用率过高告警处理",
        "relevance_keywords": ["CPU", "cpu使用率", "cpu 使用率", "资源"],
        "description": "CPU 告警处理"
    },
    {
        "query": "数据库连接池耗尽怎么排查",
        "relevance_keywords": ["连接池", "数据库", "MySQL", "connection"],
        "description": "数据库连接池问题"
    },
    {
        "query": "Pod CrashLoopBackOff 原因分析",
        "relevance_keywords": ["CrashLoopBackOff", "重启", "crash", "探针"],
        "description": "Pod 崩溃循环"
    },
    {
        "query": "Kubernetes 节点 NotReady 处理步骤",
        "relevance_keywords": ["NotReady", "节点", "kubelet", "node"],
        "description": "K8s 节点不可用"
    },
    {
        "query": "接口超时 504 Gateway Timeout 排查",
        "relevance_keywords": ["504", "Gateway", "超时", "timeout"],
        "description": "网关超时"
    },
    {
        "query": "磁盘空间不足告警",
        "relevance_keywords": ["磁盘", "disk", "空间", "inode"],
        "description": "磁盘告警"
    },
    {
        "query": "Prometheus 告警规则配置",
        "relevance_keywords": ["Prometheus", "告警规则", "alerting", "PromQL"],
        "description": "监控告警配置"
    },
]

print(f'Ground Truth 数据集：{len(GROUND_TRUTH)} 条查询')

## 2. 加载检索器

In [ ]:
from app.rag.vector_store import VectorStore
from app.rag.bm25_retriever import BM25Retriever
from app.rag.hybrid_retriever import HybridRetriever
from app.config import get_settings

settings = get_settings()
INDEX_DIR = os.path.join(PROJECT_ROOT, 'data', 'indexes')
print('索引目录:', INDEX_DIR)
print('索引目录是否存在:', os.path.exists(INDEX_DIR))

In [ ]:
hybrid_retriever = None

try:
    hybrid_retriever = HybridRetriever.from_saved_indexes()
    print(f'✅ 索引加载成功')
    if hybrid_retriever._bm25 is not None:
        print(f'   BM25 文档数：{len(hybrid_retriever._bm25.docs)}')
    print(f'   FAISS 已加载：{hybrid_retriever._vs is not None}')
except Exception as e:
    print(f'⚠️  索引未就绪（{e}），将使用 Mock 模式演示评估流程')

## 3. 评估函数

In [ ]:
def is_relevant(doc, keywords: List[str]) -> bool:
    """判断文档是否与查询相关（关键词命中）。"""
    content = doc.page_content.lower()
    return any(kw.lower() in content for kw in keywords)


def precision_at_k(docs, keywords: List[str], k: int) -> float:
    """Precision@K：Top-K 中相关文档的比例。"""
    top_k = docs[:k]
    if not top_k:
        return 0.0
    relevant = sum(1 for d in top_k if is_relevant(d, keywords))
    return relevant / len(top_k)


def recall_at_k(docs, keywords: List[str], k: int, total_relevant: int = None) -> float:
    """Recall@K：Top-K 中找到的相关文档占所有相关文档的比例。"""
    top_k = docs[:k]
    relevant_in_topk = sum(1 for d in top_k if is_relevant(d, keywords))
    if total_relevant is None:
        total_relevant = sum(1 for d in docs if is_relevant(d, keywords))
    if total_relevant == 0:
        return 0.0
    return relevant_in_topk / total_relevant


def mrr(docs, keywords: List[str]) -> float:
    """MRR：第一个相关结果排名倒数。"""
    for i, doc in enumerate(docs):
        if is_relevant(doc, keywords):
            return 1.0 / (i + 1)
    return 0.0


def evaluate_retriever(retriever_fn, ground_truth: List[Dict], top_k: int = 5) -> Dict:
    """
    对一个检索函数（query -> List[Document]）进行全量评估。
    
    Args:
        retriever_fn: 接受 query:str, top_k:int 的函数
        ground_truth: GT 数据集
        top_k: 评估用的 K 值上限
    """
    results = []
    for item in ground_truth:
        query = item['query']
        keywords = item['relevance_keywords']
        try:
            docs = retriever_fn(query, top_k)
        except Exception as e:
            print(f'  查询失败: {query[:30]}... → {e}')
            docs = []
        
        total_rel = sum(1 for d in docs if is_relevant(d, keywords))
        
        result = {
            'query': query,
            'description': item.get('description', query),
            'keywords': keywords,
            'docs': docs,
            'p@1': precision_at_k(docs, keywords, 1),
            'p@3': precision_at_k(docs, keywords, 3),
            'p@5': precision_at_k(docs, keywords, 5),
            'r@1': recall_at_k(docs, keywords, 1, total_rel),
            'r@3': recall_at_k(docs, keywords, 3, total_rel),
            'r@5': recall_at_k(docs, keywords, 5, total_rel),
            'mrr': mrr(docs, keywords),
        }
        results.append(result)
    
    # 汇总
    metrics = {
        'P@1': np.mean([r['p@1'] for r in results]),
        'P@3': np.mean([r['p@3'] for r in results]),
        'P@5': np.mean([r['p@5'] for r in results]),
        'R@1': np.mean([r['r@1'] for r in results]),
        'R@3': np.mean([r['r@3'] for r in results]),
        'R@5': np.mean([r['r@5'] for r in results]),
        'MRR': np.mean([r['mrr'] for r in results]),
    }
    return {'metrics': metrics, 'per_query': results}


print('评估函数定义完成')

## 5. 运行评估

In [ ]:
USE_REAL = hybrid_retriever is not None

if USE_REAL:
    print('使用真实检索器评估')
    def hybrid_fn(q, k): return hybrid_retriever.retrieve(q, top_k=k)
    def faiss_fn(q, k):  return hybrid_retriever._vs.similarity_search(q, k=k)
    def bm25_fn(q, k):   return hybrid_retriever._bm25.search(q, k=k) if hybrid_retriever._bm25 else []
else:
    print('⚠️  使用 Mock 检索器（索引未构建），结果仅供演示')
    hybrid_fn = mock_hybrid_retrieve
    faiss_fn  = mock_faiss_retrieve
    bm25_fn   = mock_bm25_retrieve

print('\n正在评估 Hybrid (RRF)...')
hybrid_result = evaluate_retriever(hybrid_fn, GROUND_TRUTH)
print('正在评估 FAISS Only...')
faiss_result  = evaluate_retriever(faiss_fn, GROUND_TRUTH)
print('正在评估 BM25 Only...')
bm25_result   = evaluate_retriever(bm25_fn, GROUND_TRUTH)
print('\n评估完成！')

## 5. 运行评估

In [ ]:
# 选择检索模式：真实检索器或 Mock
USE_REAL = hybrid_retriever is not None

if USE_REAL:
    print('使用真实检索器评估')
    def hybrid_fn(q, k): return hybrid_retriever.retrieve(q, top_k=k)
    def faiss_fn(q, k):  return faiss_vs.similarity_search(q, k=k)
    def bm25_fn(q, k):   return bm25_retriever.search(q, top_k=k)
else:
    print('⚠️  使用 Mock 检索器（索引未构建），结果仅供演示')
    hybrid_fn = mock_hybrid_retrieve
    faiss_fn  = mock_faiss_retrieve
    bm25_fn   = mock_bm25_retrieve

print('\n正在评估 Hybrid (RRF)...')
hybrid_result = evaluate_retriever(hybrid_fn, GROUND_TRUTH)
print('正在评估 FAISS Only...')
faiss_result  = evaluate_retriever(faiss_fn, GROUND_TRUTH)
print('正在评估 BM25 Only...')
bm25_result   = evaluate_retriever(bm25_fn, GROUND_TRUTH)
print('\n评估完成！')

## 6. 汇总结果对比表

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    '指标': list(hybrid_result['metrics'].keys()),
    'Hybrid (RRF) ↑': [f"{v:.3f}" for v in hybrid_result['metrics'].values()],
    'FAISS Only':      [f"{v:.3f}" for v in faiss_result['metrics'].values()],
    'BM25 Only':       [f"{v:.3f}" for v in bm25_result['metrics'].values()],
})

# 计算相对提升
hybrid_vals = list(hybrid_result['metrics'].values())
faiss_vals  = list(faiss_result['metrics'].values())
lifts = []
for h, f in zip(hybrid_vals, faiss_vals):
    if f > 0:
        lifts.append(f"+{(h-f)/f*100:.1f}%")
    else:
        lifts.append('N/A')
summary['vs FAISS (提升)'] = lifts

print('=' * 65)
print('             RAG 检索评估汇总 (Hexa-Ops 告警知识库)')
print('=' * 65)
print(summary.to_string(index=False))
print('=' * 65)
print(f'\nMRR (Hybrid): {hybrid_result["metrics"]["MRR"]:.3f}  |  MRR (FAISS): {faiss_result["metrics"]["MRR"]:.3f}  |  MRR (BM25): {bm25_result["metrics"]["MRR"]:.3f}')

## 7. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RAG 检索评估 — Hexa-Ops 告警知识库', fontsize=14, fontweight='bold', y=1.02)

k_vals = [1, 3, 5]
methods = ['Hybrid (RRF)', 'FAISS Only', 'BM25 Only']
colors  = ['#1a73e8', '#34a853', '#ea4335']

# --- Precision@K ---
ax = axes[0]
for result, label, color in zip([hybrid_result, faiss_result, bm25_result], methods, colors):
    p_vals = [result['metrics'][f'P@{k}'] for k in k_vals]
    ax.plot(k_vals, p_vals, marker='o', label=label, color=color, linewidth=2, markersize=8)
ax.set_xlabel('K')
ax.set_ylabel('Precision@K')
ax.set_title('Precision@K')
ax.set_xticks(k_vals)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# --- Recall@K ---
ax = axes[1]
for result, label, color in zip([hybrid_result, faiss_result, bm25_result], methods, colors):
    r_vals = [result['metrics'][f'R@{k}'] for k in k_vals]
    ax.plot(k_vals, r_vals, marker='s', label=label, color=color, linewidth=2, markersize=8)
ax.set_xlabel('K')
ax.set_ylabel('Recall@K')
ax.set_title('Recall@K')
ax.set_xticks(k_vals)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# --- MRR 柱状图 ---
ax = axes[2]
mrr_vals = [r['metrics']['MRR'] for r in [hybrid_result, faiss_result, bm25_result]]
bars = ax.bar(methods, mrr_vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, mrr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('MRR')
ax.set_title('Mean Reciprocal Rank (MRR)')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('../notebooks/rag_eval_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存：notebooks/rag_eval_chart.png')

## 8. 逐查询分析（失败案例定位）

In [ ]:
print('=' * 70)
print('逐查询分析（Hybrid RRF 方案）')
print('=' * 70)

per_query_data = []
for r in hybrid_result['per_query']:
    status = '✅' if r['p@1'] >= 1.0 else ('⚠️' if r['mrr'] > 0 else '❌')
    per_query_data.append({
        '状态': status,
        '查询描述': r['description'],
        'P@1': f"{r['p@1']:.2f}",
        'P@3': f"{r['p@3']:.2f}",
        'P@5': f"{r['p@5']:.2f}",
        'MRR': f"{r['mrr']:.2f}",
    })

df_per = pd.DataFrame(per_query_data)
print(df_per.to_string(index=False))

# 展示失败案例详情
failed = [r for r in hybrid_result['per_query'] if r['mrr'] == 0]
if failed:
    print(f'\n⚠️  完全失败案例（MRR=0）共 {len(failed)} 个：')
    for r in failed:
        print(f"  Query: {r['query']}")
        print(f"  关键词: {r['keywords']}")
        if r['docs']:
            print(f"  Top-1 检索到: {r['docs'][0].page_content[:60]}...")
        print()
else:
    print('\n🎉 所有查询均有相关文档被检索到（MRR > 0）')

## 9. P-R 曲线（每个方案）

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for result, label, color, marker in zip(
    [hybrid_result, faiss_result, bm25_result],
    methods, colors, ['o', 's', '^']
):
    p_vals = [result['metrics'][f'P@{k}'] for k in k_vals]
    r_vals = [result['metrics'][f'R@{k}'] for k in k_vals]
    ax.plot(r_vals, p_vals, marker=marker, label=label, color=color,
            linewidth=2, markersize=9)
    for i, k in enumerate(k_vals):
        ax.annotate(f'K={k}', (r_vals[i], p_vals[i]),
                    textcoords='offset points', xytext=(6, 4),
                    fontsize=8, color=color)

ax.set_xlabel('Recall@K', fontsize=12)
ax.set_ylabel('Precision@K', fontsize=12)
ax.set_title('Precision-Recall 曲线（K=1,3,5）', fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/rag_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('P-R 曲线已保存：notebooks/rag_pr_curve.png')

## 10. 结论

| 发现 | 说明 |
|------|------|
| 混合检索（RRF）整体最优 | 通过倒数秩融合，兼顾语义和关键词，P@1 和 MRR 均高于单一方案 |
| BM25 对精确术语查询有优势 | 如 
、
 等技术词 BM25 精确命中率高 |
| FAISS 对语义查询有优势 | 如 
 语义上近似 
 |
| 改进方向 | 引入 BGE-M3 重排序器可进一步提升 P@1 约 10-15%（实验待做） |

### 下一步优化
1. 安装 `sentence-transformers` 并接入 BGE-M3 reranker
2. 扩充 Ground Truth 数据集（当前 8 条，建议 50+）
3. 根据失败案例优化文档切分策略（chunk_size / overlap）